# Verify the recording

Run all cells (Run ▸ Run All Cells). Each check raises if something is wrong, so
a clean run means ClickHouse holds what `config/tables.yaml` says it should.

In [ ]:
import os
from pathlib import Path

import clickhouse_connect
import matplotlib.pyplot as plt
import numpy as np
import yaml

ch = clickhouse_connect.get_client(
    host=os.environ.get("CLICKHOUSE_HOST", "localhost"), port=8123,
    username="lab", password="lab", database="market")
config_file = next(p for p in [Path("/lab/config/tables.yaml"), Path("../config/tables.yaml")] if p.exists())
config = yaml.safe_load(config_file.read_text())["tables"]
q = ch.query_df
config

## What is in ClickHouse

In [ ]:
q("""
SELECT t.name AS table, t.total_rows AS rows, formatReadableSize(t.total_bytes) AS size,
       t.metadata_modification_time AS schema_changed
FROM system.tables AS t WHERE t.database = 'market' ORDER BY t.name
""")

## Enabled tables receive rows, disabled tables do not

Rows are inserted within a second of arriving. `funding_rate` only changes a
few times a minute, so it is checked for existence, not freshness.

In [ ]:
for table, cfg in config.items():
    enabled = cfg.get("enabled", True)
    recent = q(f"SELECT count() AS n FROM market.{table} WHERE inserted_at > now() - INTERVAL 30 SECOND")["n"][0]
    print(f"{table:14} {'enabled ' if enabled else 'disabled'}  rows in the last 30 s: {recent}")
    if not enabled:
        assert recent == 0, f"{table} is disabled but still receiving rows"
    elif table != "funding_rate":
        assert recent > 0, f"{table} is enabled but received nothing in 30 s"

## Market data is sane

In [ ]:
trades = q("""
SELECT symbol, venue, count() AS trades, min(price) AS low, max(price) AS high,
       toFloat64(argMax(price, ts_event)) AS last
FROM market.trade WHERE ts_event > now() - INTERVAL 10 MINUTE GROUP BY symbol, venue ORDER BY symbol, venue
""")
assert set(trades["venue"]) == {"BINANCE", "BYBIT"}, trades
assert (trades["low"] > 0).all()
quotes = q("""
SELECT symbol, venue, toFloat64(argMax((bid_price + ask_price) / 2, ts_event)) AS mid,
       countIf(ask_price < bid_price) AS crossed
FROM market.quote WHERE ts_event > now() - INTERVAL 10 MINUTE GROUP BY symbol, venue
""")
both = trades.merge(quotes, on=["symbol", "venue"])
both["trade_vs_mid_bps"] = (both["last"] - both["mid"]) / both["mid"] * 1e4
assert (both["trade_vs_mid_bps"].abs() < 50).all(), both
assert (both["crossed"] == 0).all(), both
both

In [ ]:
btc = q("""
SELECT toStartOfInterval(ts_event, INTERVAL 5 SECOND) AS t, concat(symbol, '.', venue) AS instrument, avg(toFloat64(price)) AS price
FROM market.trade WHERE symbol LIKE 'BTC%' AND ts_event > now() - INTERVAL 10 MINUTE GROUP BY t, instrument ORDER BY t
""")
btc.pivot(index="t", columns="instrument", values="price").plot(title="BTC trade price", figsize=(10, 4))
plt.show()

## Ingest latency: exchange receive (`ts_init`) to ClickHouse (`inserted_at`)

Measured over the last minute. Records queued while ClickHouse was down (for
example while it starts up) arrive late on purpose, so an older window can
show that backlog; the per-minute breakdown below makes it visible.

In [ ]:
lag = q("""
SELECT 'trade' AS table, quantiles(0.5, 0.99)(dateDiff('millisecond', ts_init, inserted_at)) AS ms
FROM market.trade WHERE ts_event > now() - INTERVAL 10 MINUTE AND inserted_at > now() - INTERVAL 1 MINUTE
UNION ALL
SELECT 'quote', quantiles(0.5, 0.99)(dateDiff('millisecond', ts_init, inserted_at))
FROM market.quote WHERE ts_event > now() - INTERVAL 10 MINUTE AND inserted_at > now() - INTERVAL 1 MINUTE
""")
lag[["p50_ms", "p99_ms"]] = lag["ms"].tolist()
assert (lag["p99_ms"] < 5000).all(), lag
display(lag[["table", "p50_ms", "p99_ms"]])
q("""
SELECT toStartOfMinute(inserted_at) AS minute, count() AS trades,
       quantile(0.99)(dateDiff('millisecond', ts_init, inserted_at)) AS p99_ms
FROM market.trade WHERE ts_event > now() - INTERVAL 30 MINUTE AND inserted_at > now() - INTERVAL 15 MINUTE GROUP BY minute ORDER BY minute
""")

## Book snapshots (dynamic table)

`book_snapshot` starts disabled. Set `enabled: true` for it in
`config/tables.yaml`, wait a few seconds, and re-run this cell.

In [ ]:
if q("SELECT count() AS n FROM market.book_snapshot")["n"][0] > 0:
    book = q("""
    SELECT concat(symbol, '.', venue) AS instrument, ts_event,
           arrayMap(x -> toFloat64(x), bids.price) AS `bids.price`, arrayMap(x -> toFloat64(x), bids.size) AS `bids.size`,
           arrayMap(x -> toFloat64(x), asks.price) AS `asks.price`, arrayMap(x -> toFloat64(x), asks.size) AS `asks.size`
    FROM market.book_snapshot WHERE symbol LIKE 'BTC%' ORDER BY ts_event DESC LIMIT 1
    """).iloc[0]
    assert book["bids.price"][0] < book["asks.price"][0], "crossed book"
    assert list(book["bids.price"]) == sorted(book["bids.price"], reverse=True), "bids not best-first"
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.step(book["bids.price"], np.cumsum(book["bids.size"]), label="bids")
    ax.step(book["asks.price"], np.cumsum(book["asks.size"]), label="asks")
    ax.set_title(f"{book['instrument']} cumulative depth at {book['ts_event']}")
    ax.legend()
    plt.show()
else:
    print("book_snapshot has no rows yet: enable it in config/tables.yaml")